In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os

# Función para visualizar patrones
def visualizar_patron(patron, shape=(8,8)):
    plt.imshow(patron.reshape(shape), cmap='gray')
    plt.axis('off')
    plt.show()

# Patrones 8x8 para A,C,S,T,V,M,/,+
patrones = {
    'A': np.array([
        [-1,-1, 1, 1, 1, 1,-1,-1],
        [-1, 1,-1,-1,-1,-1, 1,-1],
        [ 1,-1,-1,-1,-1,-1,-1, 1],
        [ 1, 1, 1, 1, 1, 1, 1, 1],
        [ 1,-1,-1,-1,-1,-1,-1, 1],
        [ 1,-1,-1,-1,-1,-1,-1, 1],
        [ 1,-1,-1,-1,-1,-1,-1, 1],
        [-1,-1,-1,-1,-1,-1,-1,-1]
    ]).flatten(),

    'C': np.array([
        [-1, 1, 1, 1, 1, 1, 1,-1],
        [ 1, 1,-1,-1,-1,-1,-1, 1],
        [ 1,-1,-1,-1,-1,-1,-1,-1],
        [ 1,-1,-1,-1,-1,-1,-1,-1],
        [ 1,-1,-1,-1,-1,-1,-1,-1],
        [ 1,-1,-1,-1,-1,-1,-1,-1],
        [ 1, 1,-1,-1,-1,-1,-1, 1],
        [-1, 1, 1, 1, 1, 1, 1,-1]
    ]).flatten(),

    'S': np.array([
        [ 1, 1, 1, 1, 1, 1, 1, 1],
        [ 1,-1,-1,-1,-1,-1,-1,-1],
        [ 1, 1, 1, 1, 1, 1, 1,-1],
        [-1,-1,-1,-1,-1,-1,-1, 1],
        [-1,-1,-1,-1,-1,-1,-1, 1],
        [ 1, 1, 1, 1, 1, 1, 1,-1],
        [-1,-1,-1,-1,-1,-1,-1,-1],
        [ 1, 1, 1, 1, 1, 1, 1, 1]
    ]).flatten(),

    'T': np.array([
        [ 1, 1, 1, 1, 1, 1, 1, 1],
        [-1,-1,-1, 1, 1,-1,-1,-1],
        [-1,-1,-1, 1, 1,-1,-1,-1],
        [-1,-1,-1, 1, 1,-1,-1,-1],
        [-1,-1,-1, 1, 1,-1,-1,-1],
        [-1,-1,-1, 1, 1,-1,-1,-1],
        [-1,-1,-1, 1, 1,-1,-1,-1],
        [-1,-1,-1, 1, 1,-1,-1,-1]
    ]).flatten(),

    'V': np.array([
        [ 1,-1,-1,-1,-1,-1,-1, 1],
        [ 1,-1,-1,-1,-1,-1,-1, 1],
        [ 1,-1,-1,-1,-1,-1,-1, 1],
        [ 1,-1,-1,-1,-1,-1,-1, 1],
        [-1, 1,-1,-1,-1,-1, 1,-1],
        [-1,-1, 1,-1,-1, 1,-1,-1],
        [-1,-1,-1, 1, 1,-1,-1,-1],
        [-1,-1,-1,-1,-1,-1,-1,-1]
    ]).flatten(),

    'M': np.array([
        [ 1,-1,-1,-1,-1,-1,-1, 1],
        [ 1, 1,-1,-1,-1,-1, 1, 1],
        [ 1,-1, 1,-1,-1, 1,-1, 1],
        [ 1,-1,-1, 1, 1,-1,-1, 1],
        [ 1,-1,-1,-1,-1,-1,-1, 1],
        [ 1,-1,-1,-1,-1,-1,-1, 1],
        [ 1,-1,-1,-1,-1,-1,-1, 1],
        [ 1,-1,-1,-1,-1,-1,-1, 1]
    ]).flatten(),

    '/': np.array([
        [-1,-1,-1,-1,-1,-1, 1,-1],
        [-1,-1,-1,-1,-1, 1,-1,-1],
        [-1,-1,-1,-1, 1,-1,-1,-1],
        [-1,-1,-1, 1,-1,-1,-1,-1],
        [-1,-1, 1,-1,-1,-1,-1,-1],
        [-1, 1,-1,-1,-1,-1,-1,-1],
        [ 1,-1,-1,-1,-1,-1,-1,-1],
        [-1,-1,-1,-1,-1,-1,-1,-1]
    ]).flatten(),

    '+': np.array([
        [-1,-1,-1, 1, 1,-1,-1,-1],
        [-1,-1,-1, 1, 1,-1,-1,-1],
        [-1,-1,-1, 1, 1,-1,-1,-1],
        [ 1, 1, 1, 1, 1, 1, 1, 1],
        [ 1, 1, 1, 1, 1, 1, 1, 1],
        [-1,-1,-1, 1, 1,-1,-1,-1],
        [-1,-1,-1, 1, 1,-1,-1,-1],
        [-1,-1,-1, 1, 1,-1,-1,-1]
    ]).flatten(),
}


# Entrenamiento de la red Hopfield
num_neurons = 64
weights = np.zeros((num_neurons, num_neurons))
for pattern in patrones.values():
    weights += np.outer(pattern, pattern)
np.fill_diagonal(weights, 0)

print("Red Hopfield entrenada con éxito para 8 patrones más ortogonales.")

Red Hopfield entrenada con éxito para 8 patrones más ortogonales.


In [2]:
# ======================================================
# FUNCIONES AUXILIARES
# ======================================================

def visualizar_patron_guardar(patron, shape=(8,8), titulo="", ruta_guardado=None):
    """Muestra y/o guarda un patrón binario."""
    plt.figure(figsize=(2,2))
    plt.imshow(patron.reshape(shape), cmap='gray')
    plt.axis('off')
    plt.title(titulo)
    if ruta_guardado:
        plt.savefig(ruta_guardado, bbox_inches='tight', pad_inches=0)
    plt.close()

def agregar_ruido(patron, porcentaje):
    """Invierte el signo de un porcentaje aleatorio de bits del patrón."""
    ruidoso = patron.copy()
    num_flip = int(len(patron) * porcentaje / 100)
    flip_indices = np.random.choice(range(len(patron)), size=num_flip, replace=False)
    ruidoso[flip_indices] *= -1
    return ruidoso

def recuperar_patron(patron_ruidoso, weights, pasos=8, carpeta=None, letra="", nivel=None):
    """Recupera un patrón mostrando la evolución en cada iteración y guardando imágenes."""
    rec = patron_ruidoso.copy()
    for step in range(pasos):
        for i in range(len(rec)):
            entrada = np.dot(weights[i, :], rec)
            rec[i] = 1 if entrada >= 0 else -1

        # Guardar imagen de la iteración actual
        if carpeta:
            ruta_iter = os.path.join(carpeta, f"iteracion_{step+1}.png")
            visualizar_patron_guardar(rec, (8,8),
                titulo=f"{letra} - {nivel}% ruido - iter {step+1}",
                ruta_guardado=ruta_iter
            )

    return rec

In [ ]:

patrones_prueba = ['A', 'C', '+', '/']
niveles_ruido = [15, 25, 35]

# Función para convertir nombres
def nombre_valido(s):
    reemplazos = {
        '/': 'slash',
        '+': 'plus'
    }
    return reemplazos.get(s, s)

# Carpeta base
carpeta_base = os.path.join(os.getcwd(), "resultados_hopfield")
os.makedirs(carpeta_base, exist_ok=True)

for letra in patrones_prueba:
    original = patrones[letra]
    nombre = nombre_valido(letra)

    for nivel in niveles_ruido:
        # Crear carpeta segura
        carpeta = os.path.join(carpeta_base, f"{nombre}_{nivel}porc")
        os.makedirs(carpeta, exist_ok=True)

        # Guardar patrón original
        visualizar_patron_guardar(
            original, (8,8),
            titulo=f"{letra} original",
            ruta_guardado=os.path.join(carpeta, "original.png")
        )

        # Generar patrón ruidoso
        ruidoso = agregar_ruido(original, nivel)
        visualizar_patron_guardar(
            ruidoso, (8,8),
            titulo=f"{letra} con {nivel}% ruido",
            ruta_guardado=os.path.join(carpeta, "ruidoso.png")
        )

        # Recuperación
        recuperado = recuperar_patron(
            ruidoso, weights, pasos=8,
            carpeta=carpeta, letra=letra, nivel=nivel
        )

        # Guardar resultado final
        visualizar_patron_guardar(
            recuperado, (8,8),
            titulo=f"{letra} recuperado final",
            ruta_guardado=os.path.join(carpeta, "recuperado_final.png")
        )

print("Todas las imágenes han sido guardadas en la carpeta 'resultados_hopfield'")

Todas las imágenes han sido guardadas en la carpeta 'resultados_hopfield'
